In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import rasterio
import random


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
S1_DIR = '/content/drive/MyDrive/ICS_project_data/Main_dataset/HandLabeled/S1Hand'
LABEL_DIR = '/content/drive/MyDrive/ICS_project_data/Main_dataset/HandLabeled/LabelHand'

##2. Dataset Loader (Sen1Floods11)
 This cell implements a 256x256 patch tiling with a 50-pixel overlap, as well as SAR backscatter standardization.

In [ ]:
class Sen1Floods11Dataset(Dataset):
    def __init__(self, s1_dir, label_dir, african_countries_only=False, patch_size=256, overlap=50):
        self.s1_dir = s1_dir
        self.label_dir = label_dir
        self.patch_size = patch_size
        self.overlap = overlap
        self.stride = patch_size - overlap  # 206

        all_files = [f for f in os.listdir(s1_dir) if f.endswith('.tif')]
        if african_countries_only:
            african_prefixes = ('Ghana', 'Nigeria', 'Somalia')
            self.s1_files = [f for f in all_files if f.startswith(african_prefixes)]
        else:
            self.s1_files = all_files

        # Pre-compute all (file_index, top, left) tile coordinates
        self.patches = []
        for file_idx, s1_name in enumerate(self.s1_files):
            s1_path = os.path.join(self.s1_dir, s1_name)
            with rasterio.open(s1_path) as src:
                h, w = src.height, src.width
            if h >= patch_size and w >= patch_size:
                for top in range(0, h - patch_size + 1, self.stride):
                    for left in range(0, w - patch_size + 1, self.stride):
                        self.patches.append((file_idx, top, left))

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        file_idx, top, left = self.patches[idx]
        s1_name = self.s1_files[file_idx]
        label_name = s1_name.replace('_S1Hand.tif', '_LabelHand.tif')

        s1_path = os.path.join(self.s1_dir, s1_name)
        label_path = os.path.join(self.label_dir, label_name)

        with rasterio.open(s1_path) as src:
            s1_img = src.read()  # Shape: (2, H, W)
            s1_img = np.nan_to_num(s1_img)

            # NORMALIZATION: Standardize (Z-score) each channel to help U-Net converge
            for c in range(s1_img.shape[0]):
                mean, std = np.mean(s1_img[c]), np.std(s1_img[c])
                if std > 0:
                    s1_img[c] = (s1_img[c] - mean) / std

        with rasterio.open(label_path) as src:
            label_img = src.read(1)  # Shape: (H, W)

        # PATCHING: Extract 256x256 tile with 50-pixel overlap
        s1_img = s1_img[:, top:top+self.patch_size, left:left+self.patch_size]
        label_img = label_img[top:top+self.patch_size, left:left+self.patch_size]

        s1_tensor = torch.tensor(s1_img, dtype=torch.float32)
        label_tensor = torch.tensor(label_img, dtype=torch.float32).unsqueeze(0)

        return s1_tensor, label_tensor

In [ ]:
# Instantiate dataset and output descriptive statistics
dataset = Sen1Floods11Dataset(S1_DIR, LABEL_DIR, african_countries_only=False)

print("=== Dataset Description ===")
print(f"Total base S1 geotiff images: {len(dataset.s1_files)}")
print(f"Total extracted patches (stride={dataset.stride}): {len(dataset)}")

# Inspect a sample patch to describe the shape and characteristics
s1_sample, label_sample = dataset[0]
print(f"\nSample Patch Details:")
print(f"  S1 Tensor Shape (Channels, H, W): {s1_sample.shape}")
print(f"  Label Tensor Shape (Channels, H, W): {label_sample.shape}")
print(f"  S1 Tensor Mean (after normalization): {s1_sample.mean().item():.4f}")
print(f"  S1 Tensor Std (after normalization): {s1_sample.std().item():.4f}")
print(f"  Unique classes in label: {label_sample.unique().tolist()}")

=== Dataset Description ===
Total base S1 geotiff images: 446
Total extracted patches (stride=206): 1784

Sample Patch Details:
  S1 Tensor Shape (Channels, H, W): torch.Size([2, 256, 256])
  Label Tensor Shape (Channels, H, W): torch.Size([1, 256, 256])
  S1 Tensor Mean (after normalization): 0.4014
  S1 Tensor Std (after normalization): 0.0033
  Unique classes in label: [-1.0]


## 3. U-Net Architecture
 This cell defines the U-Net model with skip connections for binary semantic segmentation.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1):
        super().__init__()
        self.down1 = DoubleConv(in_channels, 64)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(64, 128)
        self.pool2 = nn.MaxPool2d(2)
        self.down3 = DoubleConv(128, 256)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = DoubleConv(256, 512)

        self.up1 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(256, 128)
        self.up3 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(128, 64)

        self.out_conv = nn.Conv2d(64, out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))
        x_btn = self.bottleneck(self.pool3(x3))

        x = self.conv1(torch.cat([self.up1(x_btn), x3], dim=1))
        x = self.conv2(torch.cat([self.up2(x), x2], dim=1))
        x = self.conv3(torch.cat([self.up3(x), x1], dim=1))
        return self.sigmoid(self.out_conv(x))

## 4. Loss Function & Metrics
 Implements Masked BCE + Dice Loss (ignoring NoData `-1` pixels) and validation metrics.

In [ ]:
class MaskedBCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss(reduction='none')

    def forward(self, pred, target):
        # Ignore NoData (-1 pixels)
        valid_mask = (target != -1).float()
        safe_target = target.clamp(min=0) # Convert -1 to 0 temporarily for math

        # Masked BCE
        bce_loss = self.bce(pred, safe_target)
        bce_loss = (bce_loss * valid_mask).sum() / (valid_mask.sum() + 1e-8)

        # Masked Dice
        smooth = 1e-5
        pred_flat = pred.view(-1)
        target_flat = safe_target.view(-1)
        mask_flat = valid_mask.view(-1)

        intersection = (pred_flat * target_flat * mask_flat).sum()
        denominator = (pred_flat * mask_flat).sum() + (target_flat * mask_flat).sum()
        dice_loss = 1 - ((2. * intersection + smooth) / (denominator + smooth))

        return bce_loss + dice_loss

def calculate_metrics(pred, target):
    """Calculate IoU, F1-score, Precision, and Recall on valid pixels only."""
    valid_mask = (target != -1)
    preds_bin = (pred > 0.5).bool() & valid_mask
    targets_bin = (target > 0).bool() & valid_mask

    tp = (preds_bin & targets_bin).float().sum()
    fp = (preds_bin & ~targets_bin & valid_mask).float().sum()
    fn = (~preds_bin & targets_bin & valid_mask).float().sum()

    precision = (tp / (tp + fp + 1e-8)).item()
    recall = (tp / (tp + fn + 1e-8)).item()
    f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
    iou = (tp / (tp + fp + fn + 1e-8)).item()

    return iou, f1, precision, recall


## 5. Training Loop & Test Evaluation
 This cell executes the 60/20/20 split training, tracks early stopping on validation IoU, and tests the final model.


In [ ]:
def train_model():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on device: {device}")

    # Initialize Dataset
    full_dataset = Sen1Floods11Dataset(S1_DIR, LABEL_DIR, african_countries_only=False, patch_size=256, overlap=50)
    print(f"Total patches: {len(full_dataset)} (from {len(full_dataset.s1_files)} images)")

    # 60/20/20 Split
    total_len = len(full_dataset)
    train_len = int(0.6 * total_len)
    val_len = int(0.2 * total_len)
    test_len = total_len - train_len - val_len

    train_set, val_set, test_set = random_split(full_dataset, [train_len, val_len, test_len])

    train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=8, shuffle=False)

    model = UNet(in_channels=2, out_channels=1).to(device)
    criterion = MaskedBCEDiceLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    num_epochs = 100
    patience = 15
    best_iou = 0.0
    epochs_no_improve = 0

    for epoch in range(num_epochs):
        # Training Phase
        model.train()
        train_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation Phase
        model.eval()
        val_loss = 0
        val_iou, val_f1, val_prec, val_rec = 0, 0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                val_loss += criterion(outputs, labels).item()
                iou, f1, prec, rec = calculate_metrics(outputs, labels)
                val_iou += iou
                val_f1 += f1
                val_prec += prec
                val_rec += rec

        num_val_batches = len(val_loader)
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / num_val_batches
        avg_val_iou = val_iou / num_val_batches
        avg_val_f1 = val_f1 / num_val_batches
        avg_val_prec = val_prec / num_val_batches
        avg_val_rec = val_rec / num_val_batches

        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | Val IoU: {avg_val_iou:.4f} | "
              f"Val F1: {avg_val_f1:.4f} | Prec: {avg_val_prec:.4f} | Rec: {avg_val_rec:.4f}")

        # Early Stopping logic
        if avg_val_iou > best_iou:
            best_iou = avg_val_iou
            epochs_no_improve = 0
            torch.save(model.state_dict(), '/content/drive/MyDrive/unet_sen1floods_best.pth')
            print("  [*] Best model saved!")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break


    # Test Set Evaluation

    print("\n--- Test Set Evaluation ---")
    model.load_state_dict(torch.load('/content/drive/MyDrive/unet_sen1floods_best.pth'))
    model.eval()

    test_loader = DataLoader(test_set, batch_size=8, shuffle=False)
    test_iou, test_f1, test_prec, test_rec = 0, 0, 0, 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            iou, f1, prec, rec = calculate_metrics(outputs, labels)
            test_iou += iou
            test_f1 += f1
            test_prec += prec
            test_rec += rec

    num_test_batches = len(test_loader)
    print(f"Test IoU:       {test_iou / num_test_batches:.4f}")
    print(f"Test F1 (Dice): {test_f1 / num_test_batches:.4f}")
    print(f"Test Precision: {test_prec / num_test_batches:.4f}")
    print(f"Test Recall:    {test_rec / num_test_batches:.4f}")


In [ ]:
if __name__ == '__main__':
    train_model()


Training on device: cuda
Total patches: 1784 (from 446 images)
Epoch 1: Train Loss: 1.3208 | Val Loss: 1.2800 | Val IoU: 0.3024 | Val F1: 0.4350 | Prec: 0.3424 | Rec: 0.7123
  [*] Best model saved!
Epoch 2: Train Loss: 1.1570 | Val Loss: 1.2951 | Val IoU: 0.2948 | Val F1: 0.4237 | Prec: 0.3167 | Rec: 0.7816
Epoch 3: Train Loss: 1.0949 | Val Loss: 1.0248 | Val IoU: 0.3902 | Val F1: 0.5243 | Prec: 0.4540 | Rec: 0.6916
  [*] Best model saved!
Epoch 4: Train Loss: 1.0209 | Val Loss: 0.8918 | Val IoU: 0.4494 | Val F1: 0.5795 | Prec: 0.5628 | Rec: 0.6463
  [*] Best model saved!
Epoch 5: Train Loss: 0.9348 | Val Loss: 0.8314 | Val IoU: 0.4698 | Val F1: 0.6022 | Prec: 0.6329 | Rec: 0.6131
  [*] Best model saved!
Epoch 6: Train Loss: 0.9278 | Val Loss: 1.0079 | Val IoU: 0.3630 | Val F1: 0.4977 | Prec: 0.3922 | Rec: 0.7952
Epoch 7: Train Loss: 0.8666 | Val Loss: 1.1150 | Val IoU: 0.3314 | Val F1: 0.4660 | Prec: 0.3533 | Rec: 0.7932
Epoch 8: Train Loss: 0.8099 | Val Loss: 0.7063 | Val IoU: 0.4734